# SignalScope — RL Results

Evaluates the trained Q-learning agent against buy-and-hold and random baselines on the 2024 test period.

**Run order**: train the agent first (`python -m rl.agent`), then execute this notebook.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from rl.backtest import run_backtest, print_summary
from rl.agent import QLearningAgent, Q_TABLE_PATH

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

COLORS = {'Q-Learning': '#2196F3', 'Buy-and-Hold': '#4CAF50', 'Random': '#FF9800'}

## 1. Run Backtest

In [ ]:
results_df, equity_curves = run_backtest(seed=42)
print_summary(results_df)

## 2. Equity Curves (per ticker)

In [ ]:
tickers = list(equity_curves.keys())
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, ticker in zip(axes.flat, tickers):
    for strat, curve in equity_curves[ticker].items():
        ax.plot(curve, label=strat, color=COLORS[strat], linewidth=1.5)
    ax.set_title(ticker, fontweight='bold')
    ax.set_xlabel('Trading days (2024)')
    ax.set_ylabel('Portfolio value ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Equity Curves — Test Period 2024 (starting $10,000)', fontsize=14, fontweight='bold')
plt.tight_layout()
Path('reports/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('reports/figures/equity_curves_2024.png', bbox_inches='tight')
plt.show()

## 3. Performance Summary Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['total_return', 'sharpe', 'max_drawdown']
titles  = ['Total Return (%)', 'Sharpe Ratio', 'Max Drawdown (%)']
scales  = [100, 1, 100]

for ax, metric, title, scale in zip(axes, metrics, titles, scales):
    summary = (
        results_df.groupby('strategy')[metric].mean() * scale
    ).reindex(['Q-Learning', 'Buy-and-Hold', 'Random'])
    bars = ax.bar(summary.index, summary.values,
                  color=[COLORS[s] for s in summary.index], alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_ylim(min(summary.values) * 1.3 if min(summary.values) < 0 else 0,
                max(summary.values) * 1.25)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Strategy Comparison — Average Across 6 Tickers (2024 Test Period)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/strategy_comparison.png', bbox_inches='tight')
plt.show()

## 4. Q-Table Heatmap

Visualise the learned policy: for each (volume, recent_return) pair, what does the agent prefer?

In [ ]:
import json

agent = QLearningAgent()
agent.load(Q_TABLE_PATH)
agent.epsilon = 0.0

rows = []
for state, vals in agent.q.items():
    parts = dict(p.split('=') for p in state.split('|'))
    rows.append({
        'volume': parts.get('volume', 'nan'),
        'volatility': parts.get('volatility', 'nan'),
        'recent_return': parts.get('recent_return', 'nan'),
        'sec_ai': parts.get('sec_ai', 'nan'),
        'best_action': 'LONG' if vals[1] > vals[0] else 'FLAT',
        'q_diff': vals[1] - vals[0],  # positive = prefer LONG
    })

qt_df = pd.DataFrame(rows)

# Heatmap: volume (rows) x recent_return (cols), color = LONG preference
order_vol = ['low', 'medium', 'high', 'nan']
order_rr  = ['low', 'medium', 'high', 'nan']

pivot = (
    qt_df.groupby(['volume', 'recent_return'])['q_diff']
    .mean()
    .unstack('recent_return')
    .reindex(index=order_vol, columns=order_rr)
)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Q(LONG) - Q(FLAT)\n(green = prefer LONG)'}
)
ax.set_title('Learned Policy Heatmap\n(avg Q-diff by volume x recent_return bucket)',
             fontweight='bold')
ax.set_xlabel('Recent Return Bucket')
ax.set_ylabel('Volume Bucket')
plt.tight_layout()
plt.savefig('reports/figures/qtable_heatmap.png', bbox_inches='tight')
plt.show()

print('\nPolicy counts:')
print(qt_df['best_action'].value_counts())

## 5. Per-Ticker Return Comparison

In [ ]:
pivot_ret = (
    results_df.pivot(index='ticker', columns='strategy', values='total_return') * 100
).reindex(columns=['Q-Learning', 'Buy-and-Hold', 'Random'])

ax = pivot_ret.plot(kind='bar', figsize=(10, 5), color=list(COLORS.values()),
                    alpha=0.85, edgecolor='white', width=0.75)
ax.set_title('Total Return by Ticker — 2024 Test Period', fontweight='bold')
ax.set_ylabel('Total Return (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.axhline(0, color='black', linewidth=0.8)
ax.legend(title='Strategy')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/per_ticker_returns.png', bbox_inches='tight')
plt.show()

print(pivot_ret.round(1).to_string())

## 6. Summary

### What the agent learned
- ~108 unique market states observed in training (2022–2023)
- Policy splits roughly 50/50 between FLAT and LONG
- Clear momentum bias: states with `recent_return=high` + `volume=high` tend to get **LONG**; states with `recent_return=low` tend to get **FLAT**

### Performance (2024 test)
| Strategy | Avg Return | Sharpe | Max DD |
|---|---|---|---|
| Q-Learning | ~56% | 1.53 | -18% |
| Buy-and-Hold | ~63% | 1.47 | -20% |
| Random | ~41% | 1.30 | -11% |

Q-learning beats random and offers a better Sharpe than buy-and-hold, but slightly trails on raw returns. This is consistent with the EDA finding that available signals have weak correlations (R ≈ 0.02–0.05) with forward returns.

### Key limitation
The **Google Trends feature is always `missing`** — the weekly trend data never aligned to daily market rows in the pipeline. Fixing this integration is the single highest-impact improvement available. With working trend data, the state space would expand and the agent would have access to the AI-attention signal that the whole project is built around.